In [0]:
# %sql
# drop table pyspark_cata.source.customers

In [0]:
%sql
create table pyspark_cata.source.customers(
  id STRING,
  email STRING,
  city STRING,
  country STRING,
  modifiedDate TIMESTAMP
)

In [0]:
%sql
insert into pyspark_cata.source.customers
values
(1,'john@example.com','New York','USA',current_timestamp()),
(2,'jane@example.com','London','UK',current_timestamp()),
(3,'mike@example.com','Paris','France',current_timestamp()),
(4,'sara@j.com','Tokyo','Japan',current_timestamp()),
(5,'peter@t.com','Sydney','Australia',current_timestamp())

In [0]:
%sql
select * from pyspark_cata.source.customers

In [0]:
# %sql
# drop table pyspark_cata.source.dimCustomers

In [0]:
if spark.catalog.tableExists('pyspark_cata.source.dimCustomers'):
    pass

else:
    spark.sql("""
              create table pyspark_cata.source.dimCustomers
              select *,
                        current_timestamp() as startTime,
                        cast('3000-01-01' as timestamp) as endTime,
                        'Y' as isActive
              from pyspark_cata.source.customers
              """)

In [0]:
%sql
select * from pyspark_cata.source.dimCustomers

### SCD TYPE 2

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
# Source

df = spark.sql("""
               select * from pyspark_cata.source.customers
               """)

df = df.withColumn('dedup',row_number().over(Window.partitionBy('id').orderBy(desc('modifiedDate')))).drop('dedup')

df.createOrReplaceTempView('srctemp')

df = spark.sql("""
               select *,
               current_timestamp() as startTime,
               cast('3000-01-01' as timestamp) as endTime,
               'Y' as isActive
               from srctemp
               """)

df.createOrReplaceTempView('src')

### **Merge 1 - Marking the updated records as expired**

In [0]:
%sql
merge into pyspark_cata.source.dimCustomers as trg
using src as src
on trg.id = src.id 
and trg.isActive = 'Y'

WHEN MATCHED AND src.email <> trg.email 
OR src.city <> trg.city 
OR src.country <> trg.country
OR src.modifiedDate <> trg.modifiedDate

THEN UPDATE SET 
trg.endTime = current_timestamp(),
trg.isActive = 'N'

### **Merge 2 - Inserting new + updated records**

In [0]:
%sql
MERGE INTO pyspark_cata.source.dimCustomers as trg 
USING src as src
ON src.id = trg.id 
AND trg.isActive = 'Y'

WHEN NOT MATCHED
THEN INSERT *